**<h1>Electric Line Extension - Analysis 1**
#### Descriptive Data
<i>Any question regarding the notebook, please contact Robert Ford<br>
    Last Updated: 08/27/2025 | Start Development: 08/13/2025</i>
* Utility / IOU Data from PG&E, SDG&E, and SCE for 2023
* Goals: 1. Clean excel files for public downloads. See 'Clean 1' below for full details and results


In [48]:
#Import Libraries and Packages
import pandas as pd
import numpy as np
import matplotlib as plt



In [49]:
pge_2023_df = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/interim/PG&E Data_2023.xlsx", header=1)
sdge_2023_df = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/interim/SDGE Data_2023.xlsx", header=1)
sce_2023_MFNC_df = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/interim/Attachment A - 2024 Final SCE BE Annual Report.xlsx", header=4, sheet_name='Mixed-Fuel New Construction')
sce_2023_AENC_df = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/interim/Attachment A - 2024 Final SCE BE Annual Report.xlsx", header=4, sheet_name='All Electric New Construction')
sce_2023_MFU_df = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/interim/Attachment A - 2024 Final SCE BE Annual Report.xlsx", header=4, sheet_name='Mixed-Fuel Upgrades')
sce_2023_AEU_df = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/interim/Attachment A - 2024 Final SCE BE Annual Report.xlsx", header=4, sheet_name='All Electric Upgrades')

pge_2023_MFNC_df = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/interim/CPUC_Building Decarb Reporting 7.14.24 V4_CEC Summary_wav.xlsx", header=6, sheet_name='Mixed-Fuel New Construction')
pge_2023_AENC_df = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/interim/CPUC_Building Decarb Reporting 7.14.24 V4_CEC Summary_wav.xlsx", header=4, sheet_name='All Electric New Construction')
pge_2023_MFU_df = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/interim/CPUC_Building Decarb Reporting 7.14.24 V4_CEC Summary_wav.xlsx", header=4, sheet_name='Mixed Fuel Upgrades Existing')
pge_2023_AEU_df = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/interim/CPUC_Building Decarb Reporting 7.14.24 V4_CEC Summary_wav.xlsx", header=4, sheet_name='All Electric Upgrades Existing')


**CLEAN 1**

* For PG&E / SDG&E: Create dataframes for each sector ~ Residential, Nonresidential, and Combination (might include Mixed Use). 
    *   Add a 'Customer Class *' column & group by said column to create dataframe of all sectors
    *   Melt dataframes from wide to long using 'Month' & 'Customer Class *' as the IDs, variable name (columns) as 'type', values as 'counts'
    *   This results in a 624 x 4 dataframe for each IOU (2 dataframes)
* For SCE: 
    *   Using the individual sheets above, repeat steps for SCE
    *   Skip the summary rows, parse by month into blocks, concat blocks together and drop any repeated headers
    *   Group by 'Customer Class *', melt, and then concat results 
    *   This results in a 528 x 4 dataframe for each SCE sheet (4 dataframes)
* All six final dataframes are exported as excel spreadsheets and moved to the 'processed' data folder.

In [50]:
# Split the DataFrame into three separate DataFrames based on sector: residential, non-residential, and combined ~ drop NaN values
pge_2023_df_res = pge_2023_df.iloc[0:14].dropna(how='all').reset_index(drop=True)
pge_2023_df_nonres = pge_2023_df.iloc[17:31].dropna(how='all').reset_index(drop=True)
pge_2023_df_comb = pge_2023_df.iloc[34:50].dropna(how='all').reset_index(drop=True)
pge_2023_df_res
#pge_2023_df_nonres.reset_index(drop=True)
#pge_2023_df_comb.reset_index(drop=True)

,Month,Mixed Fuel Projects Requested,Mixed Fuel Projects Energized,All-Electric Projects Requested,All-Electric Projects Energized,All-Electric Upgrades exisitng- Project Requested,All-Electric Upgrades exisitng- Project Energized,Mixed Fuel Upgrades exisitng- Project Requested,Mixed Fuel Upgrades exisitng- Project Energized,Total Upgrades Requested,Total upgrades completed,Total Projects Requested,Total Projects Projects Energized,Percentage All-Electric Requested,Percentage All-electric energized,Percentage of All-Electric Upgrades Requested,Percentage of All-Electric Upgrades completed
0,Jan,291,55,687,86,568,119,4,15,572,134,978,141,70.245399,60.992908,99.300699,88.80597
1,Feb,317,88,736,146,576,205,9,13,585,218,1053,234,69.895537,62.393162,98.461538,94.036697
2,Mar,377,56,988,74,903,124,8,12,911,136,1365,130,72.380952,56.923077,99.121844,91.176471
3,Apr,375,149,1191,206,756,374,11,33,767,407,1566,355,76.05364,58.028169,98.565841,91.891892
4,May,250,134,666,184,603,433,14,37,617,470,916,318,72.707424,57.861635,97.730956,92.12766
5,Jun,276,99,695,155,326,349,6,21,332,370,971,254,71.575695,61.023622,98.192771,94.324324
6,Jul,236,102,599,157,65,257,1,15,66,272,835,259,71.736527,60.617761,98.484848,94.485294
7,Aug,225,126,604,198,48,361,NaN,35,48,396,829,324,72.858866,61.111111,100,91.161616
8,Sep,199,107,560,168,44,343,1,17,45,360,759,275,73.781291,61.090909,97.777778,95.277778
9,Oct,218,109,554,215,55,419,1,29,56,448,772,324,71.761658,66.358025,98.214286,93.526786


In [51]:
# Add a 'Customer Class *' column from SCE Annual Report to each sector dataframe
pge_2023_df_res['Customer Class *'] = 'Residential'
pge_2023_df_nonres['Customer Class *'] = 'Nonresidential'
pge_2023_df_comb['Customer Class *'] = 'Combination'

# Combine all three into one dataframe
pge_2023_df_all = pd.concat([pge_2023_df_res, pge_2023_df_nonres, pge_2023_df_comb], ignore_index=True)

# Move 'Customer Class *' to the front if you want
cols = ['Customer Class *'] + [col for col in pge_2023_df_all.columns if col != 'Customer Class *']
pge_2023_df_all = pge_2023_df_all[cols]

pge_2023_df_all

,Customer Class *,Month,Mixed Fuel Projects Requested,Mixed Fuel Projects Energized,All-Electric Projects Requested,All-Electric Projects Energized,All-Electric Upgrades exisitng- Project Requested,All-Electric Upgrades exisitng- Project Energized,Mixed Fuel Upgrades exisitng- Project Requested,Mixed Fuel Upgrades exisitng- Project Energized,Total Upgrades Requested,Total upgrades completed,Total Projects Requested,Total Projects Projects Energized,Percentage All-Electric Requested,Percentage All-electric energized,Percentage of All-Electric Upgrades Requested,Percentage of All-Electric Upgrades completed
0,Residential,Jan,291,55,687,86,568,119,4,15,572,134,978,141,70.245399,60.992908,99.300699,88.80597
1,Residential,Feb,317,88,736,146,576,205,9,13,585,218,1053,234,69.895537,62.393162,98.461538,94.036697
2,Residential,Mar,377,56,988,74,903,124,8,12,911,136,1365,130,72.380952,56.923077,99.121844,91.176471
3,Residential,Apr,375,149,1191,206,756,374,11,33,767,407,1566,355,76.05364,58.028169,98.565841,91.891892
4,Residential,May,250,134,666,184,603,433,14,37,617,470,916,318,72.707424,57.861635,97.730956,92.12766
5,Residential,Jun,276,99,695,155,326,349,6,21,332,370,971,254,71.575695,61.023622,98.192771,94.324324
6,Residential,Jul,236,102,599,157,65,257,1,15,66,272,835,259,71.736527,60.617761,98.484848,94.485294
7,Residential,Aug,225,126,604,198,48,361,NaN,35,48,396,829,324,72.858866,61.111111,100,91.161616
8,Residential,Sep,199,107,560,168,44,343,1,17,45,360,759,275,73.781291,61.090909,97.777778,95.277778
9,Residential,Oct,218,109,554,215,55,419,1,29,56,448,772,324,71.761658,66.358025,98.214286,93.526786


In [52]:
# Group by 'Customer Class *' and melt each group, then concatenate the results
melted_blocks = []
for name, group in pge_2023_df_all.groupby('Customer Class *'):
    melted = group.melt(id_vars=['Month', 'Customer Class *'], var_name='Type', value_name='Count')
    melted_blocks.append(melted)

pge_2023_df_all_Melt = pd.concat(melted_blocks, ignore_index=True)
pge_2023_df_all_Melt

,Month,Customer Class *,Type,Count
0,Jan,Combination,Mixed Fuel Projects Requested,348
1,Feb,Combination,Mixed Fuel Projects Requested,363
2,Mar,Combination,Mixed Fuel Projects Requested,430
3,Apr,Combination,Mixed Fuel Projects Requested,450
4,May,Combination,Mixed Fuel Projects Requested,298
...,...,...,...,...
619,Sep,Residential,Percentage of All-Electric Upgrades completed,95.277778
620,Oct,Residential,Percentage of All-Electric Upgrades completed,93.526786
621,Nov,Residential,Percentage of All-Electric Upgrades completed,93.312102
622,Dec,Residential,Percentage of All-Electric Upgrades completed,94.637224


In [53]:
# Split the DataFrame into three separate DataFrames based on sector: residential, non-residential, and combined ~ drop NaN values
# The sdge_2023_df_comb dataframe is only residential and non-residential, NOT mixed use.
sdge_2023_df_res = sdge_2023_df.iloc[0:14].dropna(how='all').reset_index(drop=True)
sdge_2023_df_nonres = sdge_2023_df.iloc[18:32].dropna(how='all').reset_index(drop=True)
sdge_2023_df_comb = sdge_2023_df.iloc[36:50].dropna(how='all').reset_index(drop=True)
sdge_2023_df_comb

,Month,Mixed Fuel Projects Requested,Mixed Fuel Projects Energized,All-Electric Projects Requested,All-Electric Projects Energized,Mixed Fuel upgrades Requested in Existing facilities,Mixed Fuel upgrades completed in Existing facilities,All-Electric Upgrades requested in Existing facilities,All-Electric Upgrades completed in Existing facilities,Total Upgrades requested,Total Upgrades Completed,Total Projects Requested,Total Projects Energized,Percentage All-Electric Requested,Percentage All-Electric Energized,Percentage of All-Electric Upgrades Requested,Percentage of All-Electric Upgrades completed
0,Jan,233,196,1145,94,188,173,1051,920,1239,1093,1378,1383,83.091437,6.796819,84.826473,84.172004
1,Feb,238,164,1072,67,187,124,983,722,1170,846,1310,1077,81.832061,6.220984,84.017094,85.34279
2,Mar,283,100,1329,45,223,132,1358,698,1581,830,1612,975,82.444169,4.615385,85.895003,84.096386
3,Apr,212,76,1056,25,165,118,987,605,1152,723,1268,824,83.280757,3.033981,85.677083,83.679115
4,May,217,102,1110,61,159,130,1024,533,1183,663,1327,826,83.647325,7.384988,86.559594,80.392157
5,Jun,216,226,1393,149,217,283,1294,1585,1511,1868,1609,2243,86.575513,6.642889,85.63865,84.850107
6,Jul,170,158,1252,101,127,177,1173,890,1300,1067,1422,1326,88.045007,7.616893,90.230769,83.411434
7,Aug,220,236,1520,125,194,205,1409,1405,1603,1610,1740,1971,87.356322,6.341958,87.897692,87.267081
8,Sep,241,160,1268,89,183,160,1184,864,1367,1024,1509,1273,84.029158,6.991359,86.613021,84.375
9,Oct,219,154,1095,106,175,196,1031,918,1206,1114,1314,1374,83.333333,7.714702,85.489221,82.405745


In [54]:
# Add a 'Customer Class *' column from SCE Annual Report to each sector dataframe
sdge_2023_df_res['Customer Class *'] = 'Residential'
sdge_2023_df_nonres['Customer Class *'] = 'Nonresidential'
sdge_2023_df_comb['Customer Class *'] = 'Combination'

# Combine all three into one dataframe
sdge_2023_df_all = pd.concat([sdge_2023_df_res, sdge_2023_df_nonres, sdge_2023_df_comb], ignore_index=True)

# Move 'Customer Class *' to the front if you want
cols = ['Customer Class *'] + [col for col in sdge_2023_df_all.columns if col != 'Customer Class *']
sdge_2023_df_all = sdge_2023_df_all[cols]

sdge_2023_df_all

,Customer Class *,Month,Mixed Fuel Projects Requested,Mixed Fuel Projects Energized,All-Electric Projects Requested,All-Electric Projects Energized,Mixed Fuel upgrades Requested in Existing facilities,Mixed Fuel upgrades completed in Existing facilities,All-Electric Upgrades requested in Existing facilities,All-Electric Upgrades completed in Existing facilities,Total Upgrades requested,Total Upgrades Completed,Total Projects Requested,Total Projects Energized,Percentage All-Electric Requested,Percentage All-Electric Energized,Percentage of All-Electric Upgrades Requested,Percentage of All-Electric Upgrades completed
0,Residential,Jan,143,127,1031,68,158,142,989,856,1147,998,1174,1193,87.819421,5.699916,86.224935,85.771543
1,Residential,Feb,164,112,968,42,158,111,936,685,1094,796,1132,950,85.512367,4.421053,85.557587,86.055276
2,Residential,Mar,206,100,1326,34,194,111,1295,668,1489,779,1532,913,86.553525,3.723987,86.971122,85.750963
3,Residential,Apr,141,50,952,19,143,106,933,580,1076,686,1093,755,87.099726,2.516556,86.710037,84.548105
4,Residential,May,156,67,963,43,137,111,946,503,1083,614,1119,724,86.058981,5.939227,87.349954,81.921824
5,Residential,Jun,153,174,1249,115,184,260,1233,1507,1417,1767,1402,2056,89.087019,5.593385,87.01482,85.285795
6,Residential,Jul,120,108,1121,78,111,138,1115,847,1226,985,1241,1171,90.330379,6.660974,90.946166,85.989848
7,Residential,Aug,155,176,1362,86,174,173,1332,1309,1506,1482,1517,1744,89.782465,4.931193,88.446215,88.326586
8,Residential,Sep,183,120,1148,58,160,132,1103,805,1263,937,1331,1115,86.250939,5.201794,87.33175,85.912487
9,Residential,Oct,139,105,970,76,145,163,972,859,1117,1022,1109,1203,87.466186,6.317539,87.0188,84.050881


In [55]:
# Group by 'Customer Class *' and melt each group, then concatenate the results
melted_blocks = []
for name, group in sdge_2023_df_all.groupby('Customer Class *'):
    melted = group.melt(id_vars=['Month', 'Customer Class *'], var_name='Type', value_name='Count')
    melted_blocks.append(melted)

sdge_2023_df_all_Melt = pd.concat(melted_blocks, ignore_index=True)
sdge_2023_df_all_Melt

,Month,Customer Class *,Type,Count
0,Jan,Combination,Mixed Fuel Projects Requested,233
1,Feb,Combination,Mixed Fuel Projects Requested,238
2,Mar,Combination,Mixed Fuel Projects Requested,283
3,Apr,Combination,Mixed Fuel Projects Requested,212
4,May,Combination,Mixed Fuel Projects Requested,217
...,...,...,...,...
619,Sep,Residential,Percentage of All-Electric Upgrades completed,85.912487
620,Oct,Residential,Percentage of All-Electric Upgrades completed,84.050881
621,Nov,Residential,Percentage of All-Electric Upgrades completed,88.249694
622,Dec,Residential,Percentage of All-Electric Upgrades completed,86.760125


In [56]:
# Prepare each individual sheet in the SCE 2023 data for melting
df = sce_2023_MFNC_df.copy()

monthly_blocks = []
row = 4  # Start after summary

while row < len(df):
    # Get the month label from the first cell of the row
    month_cell = df.iloc[row, 0]
    if pd.isna(month_cell):
        row += 1
        continue  # skip blank rows

    # Try to parse the month label
    try:
        month_label = pd.to_datetime(month_cell).strftime('%b %y')
    except Exception:
        row += 1
        continue  # skip if not a date

    # The next 5 rows are the data for this month (skip the header row)
    data_block = df.iloc[row+1:row+6].copy()
    data_block['Month'] = month_label
    monthly_blocks.append(data_block)

    # Move to the next block (skip the 5 data rows + 1 blank row)
    row += 6

# Combine all blocks
sce_2023_MFNC_clean = pd.concat(monthly_blocks, ignore_index=True)

# Optionally, drop rows where all columns except 'Month' are NaN (in case of trailing blanks)
sce_2023_MFNC_clean = sce_2023_MFNC_clean.dropna(subset=[col for col in sce_2023_MFNC_clean.columns if col != 'Month'], how='all')

# Move 'Month' to the front
cols = ['Month'] + [col for col in sce_2023_MFNC_clean.columns if col != 'Month']
sce_2023_MFNC_clean = sce_2023_MFNC_clean[cols]

# Drop every 5th row to match the expected number of rows
sce_2023_MFNC_clean = sce_2023_MFNC_clean.drop(sce_2023_MFNC_clean.index[::5]).reset_index(drop=True)

sce_2023_MFNC_clean

,Month,Customer Class *,Total Discounts (Non-Exempted Projects),Total Discounts (Exempted projects),Total Allowances (Non-Exempted Projects),Total Allowances (Exempted projects),Total Refund Payments Provided to Builders (Non-Exempted Projects),Total Refund Payments Provided to Builders (Exempted projects),Total Estimated Non-Refundable,Total Estimated Refundable,Total Electric Line Extension Requests Received (Applications),Total Electric Line Extensions Energized,Total Electric Line Extension Applications for Applicant Install
0,Jan 23,Residential,361982.8,0,1250226.91,0,2252901.63,0,467806.72,1912673.28,556,345,32
1,Jan 23,Industrial,0,0,0,0,0,0,0,0,0,0,0
2,Jan 23,Commercial,410316.32,0,2843419.42,0,396422.54,0,652880.76,16427.11,307,159,17
3,Jan 23,Agriculture,9098.22,0,154834.34,0,0,0,1092.75,0,16,10,0
4,Feb 24,Residential,454249.66,0,1080424.47,0,2985830.88,0,550610.99,1591659.09,603,341,11
5,Feb 24,Industrial,0,0,0,0,0,0,0,0,0,0,0
6,Feb 24,Commercial,233258.66,0,2531623.19,0,1329385.67,0,346507.42,0,300,120,11
7,Feb 24,Agriculture,8625.36,0,61999.47,0,1738.36,0,1308.15,0,14,7,0
8,Mar 23,Residential,409785.02,0,1918241.07,0,3864853.56,0,604846.16,1728677.93,693,480,24
9,Mar 23,Industrial,0,0,0,0,0,0,0,0,0,0,0


In [57]:
# Group by 'Customer Class *' and melt each group, then concatenate the results
melted_blocks = []
for name, group in sce_2023_MFNC_clean.groupby('Customer Class *'):
    melted = group.melt(id_vars=['Month', 'Customer Class *'], var_name='Type', value_name='Count')
    melted_blocks.append(melted)

sce_2023_MFNC_melted = pd.concat(melted_blocks, ignore_index=True)
sce_2023_MFNC_melted

,Month,Customer Class *,Type,Count
0,Jan 23,Agriculture,Total Discounts (Non-Exempted Projects),9098.22
1,Feb 24,Agriculture,Total Discounts (Non-Exempted Projects),8625.36
2,Mar 23,Agriculture,Total Discounts (Non-Exempted Projects),3208.13
3,Apr 23,Agriculture,Total Discounts (Non-Exempted Projects),881.12
4,May 23,Agriculture,Total Discounts (Non-Exempted Projects),0
...,...,...,...,...
523,Aug 23,Residential,Total Electric Line Extension Applications for...,13
524,Sep 23,Residential,Total Electric Line Extension Applications for...,21
525,Oct 23,Residential,Total Electric Line Extension Applications for...,32
526,Nov 23,Residential,Total Electric Line Extension Applications for...,37


In [58]:
# Prepare each individual sheet in the SCE 2023 data for melting
df2 = sce_2023_AENC_df.copy()

monthly_blocks = []
row = 4  # Start after summary

while row < len(df2):
    # Get the month label from the first cell of the row
    month_cell = df2.iloc[row, 0]
    if pd.isna(month_cell):
        row += 1
        continue  # skip blank rows

    # Try to parse the month label
    try:
        month_label = pd.to_datetime(month_cell).strftime('%b %y')
    except Exception:
        row += 1
        continue  # skip if not a date

    # The next 5 rows are the data for this month (skip the header row)
    data_block = df2.iloc[row+1:row+6].copy()
    data_block['Month'] = month_label
    monthly_blocks.append(data_block)

    # Move to the next block (skip the 5 data rows + 1 blank row)
    row += 6

# Combine all blocks
sce_2023_AENC_clean = pd.concat(monthly_blocks, ignore_index=True)

# Optionally, drop rows where all columns except 'Month' are NaN (in case of trailing blanks)
sce_2023_AENC_clean = sce_2023_AENC_clean.dropna(subset=[col for col in sce_2023_AENC_clean.columns if col != 'Month'], how='all')

# Move 'Month' to the front
cols = ['Month'] + [col for col in sce_2023_AENC_clean.columns if col != 'Month']
sce_2023_AENC_clean = sce_2023_AENC_clean[cols]

# Drop every 5th row to match the expected number of rows
sce_2023_AENC_clean = sce_2023_AENC_clean.drop(sce_2023_AENC_clean.index[::5]).reset_index(drop=True)

sce_2023_AENC_clean

,Month,Customer Class *,Total Discounts (Non-Exempted Projects),Total Discounts (Exempted projects),Total Allowances (Non-Exempted Projects),Total Allowances (Exempted projects),Total Refund Payments Provided to Builders (Non-Exempted Projects),Total Refund Payments Provided to Builders (Exempted projects),Total Estimated Non-Refundable,Total Estimated Refundable,Total Electric Line Extension Requests Received (Applications),Total Electric Line Extensions Energized,Total Electric Line Extension Applications for Applicant Install
0,Jan 23,Residential,65107.43,0,23992.77,0,0,0,24884.58,0,21,9,2
1,Jan 23,Industrial,0,0,0,0,0,0,0,0,0,0,0
2,Jan 23,Commercial,0,0,692106.78,0,0,0,46142.46,1499.97,23,14,0
3,Jan 23,Agriculture,0,0,0,0,0,0,0,0,2,0,0
4,Feb 24,Residential,0,0,20860.32,0,0,0,0,22205.04,23,9,0
5,Feb 24,Industrial,0,0,0,0,0,0,0,0,0,0,0
6,Feb 24,Commercial,0,0,1331354.53,0,0,0,80372.23,0,20,13,0
7,Feb 24,Agriculture,0,0,0,0,0,0,0,0,0,0,0
8,Mar 23,Residential,41286.15,0,60085.18,0,0,0,17435.74,0,38,13,1
9,Mar 23,Industrial,0,0,0,0,0,0,0,0,0,0,0


In [59]:
# Group by 'Customer Class *' and melt each group, then concatenate the results
melted_blocks = []
for name, group in sce_2023_AENC_clean.groupby('Customer Class *'):
    melted = group.melt(id_vars=['Month', 'Customer Class *'], var_name='Type', value_name='Count')
    melted_blocks.append(melted)

sce_2023_AENC_melted = pd.concat(melted_blocks, ignore_index=True)
sce_2023_AENC_melted

,Month,Customer Class *,Type,Count
0,Jan 23,Agriculture,Total Discounts (Non-Exempted Projects),0
1,Feb 24,Agriculture,Total Discounts (Non-Exempted Projects),0
2,Mar 23,Agriculture,Total Discounts (Non-Exempted Projects),0
3,Apr 23,Agriculture,Total Discounts (Non-Exempted Projects),0
4,May 23,Agriculture,Total Discounts (Non-Exempted Projects),8783.38
...,...,...,...,...
523,Aug 23,Residential,Total Electric Line Extension Applications for...,1
524,Sep 23,Residential,Total Electric Line Extension Applications for...,1
525,Oct 23,Residential,Total Electric Line Extension Applications for...,0
526,Nov 23,Residential,Total Electric Line Extension Applications for...,0


In [60]:
# Prepare each individual sheet in the SCE 2023 data for melting
df3 = sce_2023_MFU_df.copy()

monthly_blocks = []
row = 4  # Start after summary

while row < len(df3):
    # Get the month label from the first cell of the row
    month_cell = df3.iloc[row, 0]
    if pd.isna(month_cell):
        row += 1
        continue  # skip blank rows

    # Try to parse the month label
    try:
        month_label = pd.to_datetime(month_cell).strftime('%b %y')
    except Exception:
        row += 1
        continue  # skip if not a date

    # The next 5 rows are the data for this month (skip the header row)
    data_block = df3.iloc[row+1:row+6].copy()
    data_block['Month'] = month_label
    monthly_blocks.append(data_block)

    # Move to the next block (skip the 5 data rows + 1 blank row)
    row += 6

# Combine all blocks
sce_2023_MFU_clean = pd.concat(monthly_blocks, ignore_index=True)

# Optionally, drop rows where all columns except 'Month' are NaN (in case of trailing blanks)
sce_2023_MFU_clean = sce_2023_MFU_clean.dropna(subset=[col for col in sce_2023_MFU_clean.columns if col != 'Month'], how='all')

# Move 'Month' to the front
cols = ['Month'] + [col for col in sce_2023_MFU_clean.columns if col != 'Month']
sce_2023_MFU_clean = sce_2023_MFU_clean[cols]

# Drop every 5th row to match the expected number of rows
sce_2023_MFU_clean = sce_2023_MFU_clean.drop(sce_2023_MFU_clean.index[::5]).reset_index(drop=True)

sce_2023_MFU_clean

,Month,Customer Class *,Total Discounts (Non-Exempted Projects),Total Discounts (Exempted projects),Total Allowances (Non-Exempted Projects),Total Allowances (Exempted projects),Total Refund Payments Provided to Builders (Non-Exempted Projects),Total Refund Payments Provided to Builders (Exempted projects),Total Estimated Non-Refundable,Total Estimated Refundable,Total Electric Line Extension Requests Received (Applications),Total Electric Line Extensions Energized,Total Electric Line Extension Applications for Applicant Install
0,Jan 23,Residential,0,0,288372.5,0,0,0,0,0,816,415,0
1,Jan 23,Industrial,0,0,0,0,0,0,0,0,0,0,0
2,Jan 23,Commercial,0,0,434382.0,0,0,0,0,0,63,28,0
3,Jan 23,Agriculture,0,0,91770.09,0,0,0,0,0,15,9,0
4,Feb 24,Residential,0,0,274458.77,0,0,0,0,0,872,411,0
5,Feb 24,Industrial,0,0,0,0,0,0,0,0,0,0,0
6,Feb 24,Commercial,0,0,646380.56,0,0,0,0,0,72,33,0
7,Feb 24,Agriculture,0,0,281569.33,0,0,0,0,0,18,10,0
8,Mar 23,Residential,0,0,348657.16,0,0,0,0,0,979,519,0
9,Mar 23,Industrial,0,0,0,0,0,0,0,0,0,0,0


In [61]:
# Group by 'Customer Class *' and melt each group, then concatenate the results
melted_blocks = []
for name, group in sce_2023_MFU_clean.groupby('Customer Class *'):
    melted = group.melt(id_vars=['Month', 'Customer Class *'], var_name='Type', value_name='Count')
    melted_blocks.append(melted)

sce_2023_MFU_melted = pd.concat(melted_blocks, ignore_index=True)
sce_2023_MFU_melted

,Month,Customer Class *,Type,Count
0,Jan 23,Agriculture,Total Discounts (Non-Exempted Projects),0
1,Feb 24,Agriculture,Total Discounts (Non-Exempted Projects),0
2,Mar 23,Agriculture,Total Discounts (Non-Exempted Projects),0
3,Apr 23,Agriculture,Total Discounts (Non-Exempted Projects),0
4,May 23,Agriculture,Total Discounts (Non-Exempted Projects),0
...,...,...,...,...
523,Aug 23,Residential,Total Electric Line Extension Applications for...,0
524,Sep 23,Residential,Total Electric Line Extension Applications for...,0
525,Oct 23,Residential,Total Electric Line Extension Applications for...,0
526,Nov 23,Residential,Total Electric Line Extension Applications for...,1


In [62]:
# Prepare each individual sheet in the SCE 2023 data for melting
df4 = sce_2023_AEU_df.copy()

monthly_blocks = []
row = 4  # Start after summary

while row < len(df4):
    # Get the month label from the first cell of the row
    month_cell = df4.iloc[row, 0]
    if pd.isna(month_cell):
        row += 1
        continue  # skip blank rows

    # Try to parse the month label
    try:
        month_label = pd.to_datetime(month_cell).strftime('%b %y')
    except Exception:
        row += 1
        continue  # skip if not a date

    # The next 5 rows are the data for this month (skip the header row)
    data_block = df4.iloc[row+1:row+6].copy()
    data_block['Month'] = month_label
    monthly_blocks.append(data_block)

    # Move to the next block (skip the 5 data rows + 1 blank row)
    row += 6

# Combine all blocks
sce_2023_AEU_clean = pd.concat(monthly_blocks, ignore_index=True)

# Optionally, drop rows where all columns except 'Month' are NaN (in case of trailing blanks)
sce_2023_AEU_clean = sce_2023_AEU_clean.dropna(subset=[col for col in sce_2023_AEU_clean.columns if col != 'Month'], how='all')

# Move 'Month' to the front
cols = ['Month'] + [col for col in sce_2023_AEU_clean.columns if col != 'Month']
sce_2023_AEU_clean = sce_2023_AEU_clean[cols]

# Drop every 5th row to match the expected number of rows
sce_2023_AEU_clean = sce_2023_AEU_clean.drop(sce_2023_AEU_clean.index[::5]).reset_index(drop=True)

sce_2023_AEU_clean

,Month,Customer Class *,Total Discounts (Non-Exempted Projects),Total Discounts (Exempted projects),Total Allowances (Non-Exempted Projects),Total Allowances (Exempted projects),Total Refund Payments Provided to Builders (Non-Exempted Projects),Total Refund Payments Provided to Builders (Exempted projects),Total Estimated Non-Refundable,Total Estimated Refundable,Total Electric Line Extension Requests Received (Applications),Total Electric Line Extensions Energized,Total Electric Line Extension Applications for Applicant Install
0,Jan 23,Residential,0,0,6485.06,0,0,0,0,0,5,3,0
1,Jan 23,Industrial,0,0,0,0,0,0,0,0,0,0,0
2,Jan 23,Commercial,0,0,128253.7,0,0,0,0,0,0,4,0
3,Jan 23,Agriculture,0,0,0,0,0,0,0,0,0,0,0
4,Feb 24,Residential,0,0,539.61,0,0,0,0,0,3,1,0
5,Feb 24,Industrial,0,0,0,0,0,0,0,0,0,0,0
6,Feb 24,Commercial,0,0,0,0,0,0,0,0,0,0,0
7,Feb 24,Agriculture,0,0,0,0,0,0,0,0,0,0,0
8,Mar 23,Residential,0,0,4434.74,0,0,0,0,0,7,4,0
9,Mar 23,Industrial,0,0,0,0,0,0,0,0,0,0,0


In [63]:
# Group by 'Customer Class *' and melt each group, then concatenate the results
melted_blocks = []
for name, group in sce_2023_AEU_clean.groupby('Customer Class *'):
    melted = group.melt(id_vars=['Month', 'Customer Class *'], var_name='Type', value_name='Count')
    melted_blocks.append(melted)

sce_2023_AEU_melted = pd.concat(melted_blocks, ignore_index=True)
sce_2023_AEU_melted

,Month,Customer Class *,Type,Count
0,Jan 23,Agriculture,Total Discounts (Non-Exempted Projects),0
1,Feb 24,Agriculture,Total Discounts (Non-Exempted Projects),0
2,Mar 23,Agriculture,Total Discounts (Non-Exempted Projects),0
3,Apr 23,Agriculture,Total Discounts (Non-Exempted Projects),0
4,May 23,Agriculture,Total Discounts (Non-Exempted Projects),0
...,...,...,...,...
523,Aug 23,Residential,Total Electric Line Extension Applications for...,0
524,Sep 23,Residential,Total Electric Line Extension Applications for...,0
525,Oct 23,Residential,Total Electric Line Extension Applications for...,0
526,Nov 23,Residential,Total Electric Line Extension Applications for...,0


In [ ]:
# Export the final melted DataFrames to Excel files, verify, and move to processed folder
# Uncomment to run / reprocess specific files

#sdge_2023_df_all_Melt.to_excel('SDGE Line Extension 2023.xlsx', sheet_name='SDGE Line Extension Combined 2023', index=False)


In [66]:
#Combine all SCE melted dataframes into one composite dataframe and add IoU and Fuel Structure Type columns

sce_2023_AEU_melted['Fuel_Structure_Type'] = 'All Electric Upgrades'
sce_2023_MFU_melted['Fuel_Structure_Type'] = 'Mixed Fuel Upgrades'
sce_2023_MFNC_melted['Fuel_Structure_Type'] = 'Mixed Fuel New Construction'
sce_2023_AENC_melted['Fuel_Structure_Type'] = 'All Electric New Construction'

sce_2023_composite = pd.concat([sce_2023_AEU_melted, sce_2023_MFU_melted, sce_2023_MFNC_melted, sce_2023_AENC_melted], ignore_index=True)
sce_2023_composite['IoU'] = 'SCE'
sce_2023_composite

,Month,Customer Class *,Type,Count,Fuel_Structure_Type,IoU
0,Jan 23,Agriculture,Total Discounts (Non-Exempted Projects),0,All Electric Upgrades,SCE
1,Feb 24,Agriculture,Total Discounts (Non-Exempted Projects),0,All Electric Upgrades,SCE
2,Mar 23,Agriculture,Total Discounts (Non-Exempted Projects),0,All Electric Upgrades,SCE
3,Apr 23,Agriculture,Total Discounts (Non-Exempted Projects),0,All Electric Upgrades,SCE
4,May 23,Agriculture,Total Discounts (Non-Exempted Projects),0,All Electric Upgrades,SCE
...,...,...,...,...,...,...
2107,Aug 23,Residential,Total Electric Line Extension Applications for...,1,All Electric New Construction,SCE
2108,Sep 23,Residential,Total Electric Line Extension Applications for...,1,All Electric New Construction,SCE
2109,Oct 23,Residential,Total Electric Line Extension Applications for...,0,All Electric New Construction,SCE
2110,Nov 23,Residential,Total Electric Line Extension Applications for...,0,All Electric New Construction,SCE


In [21]:
pge_2023_AEU_df.head(20)

,Customer Class,Total Discounts (Non-Exempted Projects),Total Discounts (Exempted projects),Total Allowances (Non-Exempted Projects),Total Allowances (Exempted projects),Total Refund Payments Provided to Builders (Non-Exempted Projects),Total Refund Payments Provided to Builders (Exempted projects),Total Estimated Non-Refundable,Total Estimated Refundable,Total Electric Line Extension Requests Received (Applications),Total Electric Line Extensions Energized,Total Electric Line Extension Applications for Applicant Install
0,"Agency (City, County, Caltrans)",1698.942802,0,84530.353197,0,NaN,0,10127.0132,87928.2388,5,2,3
1,Agricultural,62130.891628,0,738253.833145,0,2115.22,0,25090.684,862515.6164,6,9,3
2,Commercial,470418.428958,0,479026.382085,0,28923.07,0,162489.0644,1419863.24,30,17,17
3,Industrial,1820.63242,0,18488.097561,0,NaN,0,125,22129.3624,5,NaN,5
4,Mixed Use (Commercial / Residential),126788.154248,0,56679.763904,0,782.71,0,22809.068,310256.0724,3,7,2
5,Residential,122971.201572,0,32074.388456,0,NaN,0,931829.0576,278016.7916,568,119,465
6,Street and Outdoor area Lighting,NaN,0,NaN,0,NaN,0,10501.6096,NaN,1,NaN,1
7,Telecommunications,3265.666016,0,712.747967,0,NaN,0,2615.6268,7244.08,15,NaN,5
8,Temporary Services,NaN,0,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
# Redo of the 2023 PGE report using the CPUC data ~ this is the correct data to use for 2023
def extract_month_and_clean(df, valid_classes, jan_label="Jan 2023", n_jan_rows=9):
    header_name = df.columns[0]
    rows = []
    current_month = None
    row_iter = df.iterrows()
    row_idx = 0

    # Handle January (first n_jan_rows rows)
    for _ in range(n_jan_rows):
        idx, row = next(row_iter)
        first_val = row[header_name]
        if first_val in valid_classes:
            new_row = row.copy()
            new_row['Month'] = jan_label
            rows.append(new_row)
        row_idx += 1

    # Handle the rest of the months
    for _, row in list(row_iter):
        first_val = row[header_name]
        # If this row is a date, update current_month
        try:
            month_val = pd.to_datetime(first_val, errors='raise')
            current_month = month_val.strftime('%b %Y')
            continue  # skip the date row itself
        except Exception:
            pass
        # Skip repeated headers
        if first_val == header_name:
            continue
        # Only keep valid customer class rows
        if first_val in valid_classes and current_month is not None:
            new_row = row.copy()
            new_row['Month'] = current_month
            rows.append(new_row)
    # Create DataFrame and reorder columns
    if rows:
        clean_df = pd.DataFrame(rows)
        cols = ['Month'] + [col for col in clean_df.columns if col != 'Month']
        clean_df = clean_df[cols].reset_index(drop=True)
        return clean_df
    else:
        return pd.DataFrame()  # empty if nothing found


In [23]:
valid_classes=['Agency (City, County, Caltrans)', 'Agricultural','Commercial', 'Industrial',
'Mixed Use (Commercial / Residential)', 'Residential', 'Street and Outdoor area Lighting',
'Telecommunications', 'Temporary Services']

pge_2023_MFNC_clean = extract_month_and_clean(pge_2023_MFNC_df,valid_classes, jan_label="Jan 2023", n_jan_rows=9)
pge_2023_AENC_clean = extract_month_and_clean(pge_2023_AENC_df,valid_classes, jan_label="Jan 2023", n_jan_rows=9)
pge_2023_MFU_clean = extract_month_and_clean(pge_2023_MFU_df,valid_classes, jan_label="Jan 2023", n_jan_rows=9)
pge_2023_AEU_clean = extract_month_and_clean(pge_2023_AEU_df,valid_classes, jan_label="Jan 2023", n_jan_rows=9)

In [35]:
def melt_by_customer_class(df, customer_class_col='Customer Class', month_col='Month'):
    """
    Groups by customer class and melts each group, then concatenates the results.
    """
    melted_blocks = []
    for name, group in df.groupby(customer_class_col):
        melted = group.melt(id_vars=[month_col, customer_class_col], var_name='Type', value_name='Count')
        melted_blocks.append(melted)
    return pd.concat(melted_blocks, ignore_index=True)

In [36]:
pge_2023_AEU_melted = melt_by_customer_class(pge_2023_AEU_clean)
pge_2023_MFU_melted = melt_by_customer_class(pge_2023_MFU_clean)
pge_2023_MFNC_melted = melt_by_customer_class(pge_2023_MFNC_clean)
pge_2023_AENC_melted = melt_by_customer_class(pge_2023_AENC_clean)

In [42]:
pge_2023_AEU_melted['Fuel_Structure_Type'] = 'All Electric Upgrades'
pge_2023_MFU_melted['Fuel_Structure_Type'] = 'Mixed Fuel Upgrades'
pge_2023_MFNC_melted['Fuel_Structure_Type'] = 'Mixed Fuel New Construction'
pge_2023_AENC_melted['Fuel_Structure_Type'] = 'All Electric New Construction'


In [46]:
pge_2023_composite = pd.concat([pge_2023_AEU_melted, pge_2023_MFU_melted, pge_2023_MFNC_melted, pge_2023_AENC_melted], ignore_index=True)
pge_2023_composite['IoU'] = 'PG&E'
pge_2023_composite

,Month,Customer Class,Type,Count,Fuel_Structure_Type,IoU
0,Jan 2023,"Agency (City, County, Caltrans)",Total Discounts (Non-Exempted Projects),1698.942802,All Electric Upgrades,PG&E
1,Feb 2024,"Agency (City, County, Caltrans)",Total Discounts (Non-Exempted Projects),83953.139397,All Electric Upgrades,PG&E
2,Mar 2023,"Agency (City, County, Caltrans)",Total Discounts (Non-Exempted Projects),13537.162969,All Electric Upgrades,PG&E
3,Apr 2023,"Agency (City, County, Caltrans)",Total Discounts (Non-Exempted Projects),119076.222584,All Electric Upgrades,PG&E
4,May 2023,"Agency (City, County, Caltrans)",Total Discounts (Non-Exempted Projects),16430.480828,All Electric Upgrades,PG&E
...,...,...,...,...,...,...
4747,Aug 2023,Temporary Services,Total Electric Line Extension Applications for...,NaN,All Electric New Construction,PG&E
4748,Sep 2023,Temporary Services,Total Electric Line Extension Applications for...,NaN,All Electric New Construction,PG&E
4749,Oct 2023,Temporary Services,Total Electric Line Extension Applications for...,NaN,All Electric New Construction,PG&E
4750,Nov 2023,Temporary Services,Total Electric Line Extension Applications for...,NaN,All Electric New Construction,PG&E


In [67]:
# Export the final composite DataFrames to Excel files, verify, and move to processed folder
# Uncomment to run / reprocess specific files

pge_2023_composite.to_excel('PGE 2023 Composite.xlsx', sheet_name='PGE Line Extension Combined 2023', index=False)
sce_2023_composite.to_excel('SCE 2023 Composite.xlsx', sheet_name='SCE Line Extension Combined 2023', index=False)


C:\Users\Rford\AppData\Roaming\Python\Python312\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
